## Setup

### Configuration

| Parameter | Value |
|-----------|-------|
| Apache Spark version | 4.0.2 |
| Apache Iceberg version | 1.10.0 |
| Number of Executors | 2 |
| Executor memory | 8GB |
| Cores per executor | 2 |
| Data size | 30GB |

In [ ]:
# Stop any existing session
try:
    spark.stop()
except:
    pass

In [ ]:
workshop_name = "file-size-deep-dive"

from pyspark.sql import SparkSession

executor_memory = "8g"
executor_cores = 2
num_executors = 2


spark = SparkSession.builder \
        .appName(workshop_name) \
        .master("spark://spark-master:7077") \
        .config("spark.executor.memory", executor_memory) \
        .config("spark.executor.cores", executor_cores) \
        .config("spark.executor.instances", num_executors) \
        .config("spark.cores.max", executor_cores * num_executors) \
    .getOrCreate()

* Open [http://localhost:4040](http://localhost:4040) to see your Spark UI

In [ ]:
# Create Fake Data 
import os 
import glob
from pathlib import Path

from generate_data import run
from run_ddl import run_ddl 

################################################################################
## CHANGE DATA SIZE AS NEEDED
################################################################################

data_size_gb = 30
overwrite_existing_data = False # CHANGE TO True TO CREATE DATA
data_folder = './data'

# Only when overwrite = False and there are atleast 1 csv files we need to skip data gen (everyother time we need to generate data) 
if not (not overwrite_existing_data and len(glob.glob(os.path.join(data_folder, '*.csv'))) > 1):
    print(f'Creating raw TPCH data in {data_folder}')
    run(scaling_factor=data_size_gb)

print(f'Loading data in raw TPCH folder {data_folder} to Iceberg Tables')
run_ddl(data_path=Path(data_folder), spark=spark, recreate=True) # Load created data into Iceberg tables on Minio (S3 equivalent)

In [ ]:
%%sql
use prod.db

## Many small files => Spark wastes time opening file

In [ ]:
%%sql
SELECT
  MONTH (l_receiptdate) AS receipt_month,
  COUNT(*) AS num_line_items
FROM lineitem
GROUP BY 1 ORDER BY 2 desc LIMIT 10

## Maintenance functions are the simplest fix

In [ ]:
%%sql
-- new demo table
CREATE TABLE prod.db.lineitem_resized  AS SELECT * FROM prod.db.lineitem;

In [ ]:
# call maintenance function on the newly created table
spark.sql("CALL demo.system.rewrite_data_files('prod.db.lineitem_resized')")

In [ ]:
%%sql
SELECT
  MONTH (l_receiptdate) AS receipt_month,
  COUNT(*) AS num_line_items
FROM lineitem_resized
GROUP BY 1 ORDER BY 2 desc LIMIT 10

## Tune data size per writer task for optimally sized output 

In [ ]:
%%sql
SELECT
  ROW_NUMBER() OVER (ORDER BY file_path) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM prod.db.lineitem.files
ORDER BY 2, 3
LIMIT 4

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_2gb_partsize (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648' -- 2 GB
  );



In [ ]:
%%sql
INSERT INTO
  prod.db.lineitem_2gb_partsize
SELECT *
FROM prod.db.lineitem

In [ ]:
%%sql
-- Let's inspect the file sizes.
SELECT
  ROW_NUMBER() OVER (ORDER BY file_path) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM prod.db.lineitem_2gb_partsize.files
ORDER BY 2, 3

## Partition data before insert

In [ ]:
%%sql
CREATE TABLE
  IF NOT EXISTS prod.db.lineitem_part_year_2gb_intask_mem (
    l_orderkey BIGINT,
    l_partkey BIGINT,
    l_suppkey BIGINT,
    l_linenumber INT,
    l_quantity DECIMAL(15, 2),
    l_extendedprice DECIMAL(15, 2),
    l_discount DECIMAL(15, 2),
    l_tax DECIMAL(15, 2),
    l_returnflag STRING,
    l_linestatus STRING,
    l_shipdate DATE,
    l_commitdate DATE,
    l_receiptdate DATE,
    l_shipinstruct STRING,
    l_shipmode STRING,
    l_comment STRING
  ) USING iceberg PARTITIONED BY (YEAR (l_shipdate)) TBLPROPERTIES (
    'format-version' = '2',
    'write.spark.advisory-partition-size-bytes' = '2147483648' -- 2 GB
  );

In [ ]:
%%sql
INSERT INTO
  prod.db.lineitem_part_year_2gb_intask_mem
SELECT
  *
FROM
  prod.db.lineitem;

In [ ]:
%%sql
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      file_path
  ) AS row_num,
  file_path,
  ROUND(file_size_in_bytes / (1024 * 1024), 2) AS file_size_mb,
  split(file_path, '-')[1] AS writer_task
FROM
  prod.db.lineitem_part_year_2gb_intask_mem.files
ORDER BY
  2, 3

In [ ]:
# This forces same-year rows to the same Icebreg write task
import pyspark.sql.functions as F

spark.table("prod.db.lineitem")\
  .repartition(F.year(F.col("l_shipdate"))) \
  .writeTo("prod.db.lineitem_part_year_2gb_intask_mem") \
  .createOrReplace()